# Técnica Supervisionada — MLP (Multi-Layer Perceptron)
**Raças:** Shiba Inu, Saint Bernard, Egyptian Mau, Russian Blue

### Metodologia
1. Para cada uma das 12 bases, executa-se o GridSearch com **Holdout 70/30** para encontrar as 10 melhores configurações (por F1-score).
2. Em seguida, as 10 melhores configurações são reavaliadas com **10-fold CV** na mesma base.
3. Os resultados são organizados na planilha MLP.

---
#### Nota sobre `hidden_layer_sizes`
A metodologia da aula prática de MLP sugere calcular o número de neurônios com base no número de atributos e classes:
- Fórmula orientativa: `n_neurônios ≈ (n_atributos + n_classes) / 2` ou variações.
- Como as bases variam muito em número de atributos, usa-se uma lista fixa de tamanhos comuns (configuração do professor) mais adaptações por base.

## 0. Configurações Globais e Imports

In [2]:
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import f1_score

# -------------------------------------------------------
# Caminho para as bases (ajuste se necessário)
# -------------------------------------------------------
BASE_PATH = './12Bases/'   # ex: '12bases/' ou ''

# Lista das 12 bases (mesmas selecionadas na atividade de k-NN)
bases = [
    'HOG_128_16x16.csv',
    'HOG_128_32x32.csv',
    'HOG_256_32x32.csv',
    'LBP_256_3R.csv',
    'LBP_256_6R.csv',
    'LBP_256_12R.csv',
    'PCA_075_HOG_128_16x16.csv',
    'PCA_075_HOG_256_8x8.csv',
    'PCA_075_HOG_256_16x16.csv',
    'PCA_075_HOG_256_32x32.csv',
    'PCA_090_HOG_128_16x16.csv',
    'PCA_090_HOG_256_32x32.csv'
]

# -------------------------------------------------------
# Hiperparâmetros do GridSearch (igual ao código do professor)
# -------------------------------------------------------
par_hls             = [(100,), (120,), (150,), (200,), (250,), (70,80), (100,100), (150,100)]
par_activation      = ['relu']          # fixar uma função de ativação
par_solver          = ['adam', 'sgd']
par_learning_rate   = [0.0001, 0.001, 0.01, 0.1]
par_max_iter        = [1000, 1500, 2000, 2500, 3000]

N_TOP = 10          # número de melhores configurações
RANDOM_STATE = 19

print('Configuração pronta.')
print(f'Total de combinações por base: {len(par_hls)*len(par_activation)*len(par_solver)*len(par_learning_rate)*len(par_max_iter)}')

Configuração pronta.
Total de combinações por base: 320


## 1. Função Principal — GridSearch Holdout + 10-fold CV

In [3]:
def run_mlp_base(nome_base, base_path=''):
    """
    Para uma dada base:
      1. Roda o GridSearch com Holdout 70/30 e coleta todos os F1-scores.
      2. Seleciona as 10 melhores configurações.
      3. Avalia as 10 melhores configurações com 10-fold CV.
    Retorna: (top10_holdout_df, top10_cv_scores_list)
    """
    print(f'\n{'='*60}')
    print(f'BASE: {nome_base}')
    print('='*60)

    # --- Carregar dados ---
    df = pd.read_csv(f'{base_path}{nome_base}', encoding='utf-8')
    X = df.iloc[:, :-1]
    y = df.iloc[:, -1]
    n_atributos = X.shape[1]
    print(f'Atributos: {n_atributos} | Amostras: {len(df)}')

    # --- ETAPA 1: GridSearch com Holdout 70/30 ---
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=RANDOM_STATE
    )

    resultados_holdout = []
    start = time.time()

    for hls in par_hls:
        for act in par_activation:
            for solv in par_solver:
                for lr in par_learning_rate:
                    for it in par_max_iter:
                        mlp = MLPClassifier(
                            hidden_layer_sizes=hls,
                            activation=act,
                            solver=solv,
                            learning_rate_init=lr,
                            max_iter=it,
                            random_state=RANDOM_STATE
                        )
                        mlp.fit(X_train, y_train)
                        y_pred = mlp.predict(X_test)
                        f1 = f1_score(y_test, y_pred, average='weighted')

                        resultados_holdout.append({
                            'f1_holdout': f1,
                            'hls': hls,
                            'activation': act,
                            'solver': solv,
                            'learning_rate': lr,
                            'max_iter': it
                        })

    elapsed = time.time() - start
    print(f'GridSearch Holdout concluído em {elapsed:.1f}s')

    # Selecionar top 10
    df_hold = pd.DataFrame(resultados_holdout)
    top10 = df_hold.nlargest(N_TOP, 'f1_holdout').reset_index(drop=True)

    print(f'\n--- Top {N_TOP} configurações (Holdout 70/30) ---')
    for i, row in top10.iterrows():
        print(f"{i+1:2d} | F1={row['f1_holdout']:.3f} | hls={row['hls']} | act={row['activation']} | "
              f"solver={row['solver']} | lr={row['learning_rate']} | iter={int(row['max_iter'])}")

    # --- ETAPA 2: 10-fold CV para as top 10 configurações ---
    kf = KFold(n_splits=10, random_state=1, shuffle=True)
    cv_scores = []

    print(f'\n--- Resultados 10-fold CV (top {N_TOP} configurações) ---')
    for i, row in top10.iterrows():
        mlp = MLPClassifier(
            hidden_layer_sizes=row['hls'],
            activation=row['activation'],
            solver=row['solver'],
            learning_rate_init=row['learning_rate'],
            max_iter=int(row['max_iter']),
            random_state=RANDOM_STATE
        )
        scores = cross_val_score(mlp, X, y, scoring='f1_weighted', cv=kf)
        media = scores.mean()
        cv_scores.append(media)
        print(f"Conf. {i+1:02d} | F1_CV={media:.3f} | hls={row['hls']} | solver={row['solver']} | "
              f"lr={row['learning_rate']} | iter={int(row['max_iter'])}")

    return top10, cv_scores

print('Função definida.')

Função definida.


## 2. Executar para todas as 12 bases

> ⚠️ **Atenção**: este processo pode demorar bastante (dezenas de minutos a horas dependendo do hardware). Execute célula por célula para acompanhar o progresso ou use um ambiente com GPU/mais CPU.

In [4]:
# Dicionário para armazenar os resultados de todas as bases
resultados_globais = {}   # {nome_base: {'top10': df, 'cv_scores': list}}

start_total = time.time()

for nome_base in bases:
    top10, cv_scores = run_mlp_base(nome_base, base_path=BASE_PATH)
    resultados_globais[nome_base] = {'top10': top10, 'cv_scores': cv_scores}

elapsed_total = time.time() - start_total
print(f'\n\nTODAS AS BASES CONCLUÍDAS em {elapsed_total/60:.1f} minutos.')


BASE: HOG_128_16x16.csv
Atributos: 1764 | Amostras: 656
GridSearch Holdout concluído em 5144.4s

--- Top 10 configurações (Holdout 70/30) ---
 1 | F1=0.735 | hls=(70, 80) | act=relu | solver=sgd | lr=0.1 | iter=1000
 2 | F1=0.735 | hls=(70, 80) | act=relu | solver=sgd | lr=0.1 | iter=1500
 3 | F1=0.735 | hls=(70, 80) | act=relu | solver=sgd | lr=0.1 | iter=2000
 4 | F1=0.735 | hls=(70, 80) | act=relu | solver=sgd | lr=0.1 | iter=2500
 5 | F1=0.735 | hls=(70, 80) | act=relu | solver=sgd | lr=0.1 | iter=3000
 6 | F1=0.732 | hls=(70, 80) | act=relu | solver=sgd | lr=0.001 | iter=1500
 7 | F1=0.732 | hls=(70, 80) | act=relu | solver=sgd | lr=0.001 | iter=2000
 8 | F1=0.732 | hls=(70, 80) | act=relu | solver=sgd | lr=0.001 | iter=2500
 9 | F1=0.732 | hls=(70, 80) | act=relu | solver=sgd | lr=0.001 | iter=3000
10 | F1=0.731 | hls=(100, 100) | act=relu | solver=adam | lr=0.0001 | iter=1000

--- Resultados 10-fold CV (top 10 configurações) ---
Conf. 01 | F1_CV=0.672 | hls=(70, 80) | solver=sg

## 3. Montar a tabela de resultados (formato planilha MLP)

In [5]:
# Mapeamento base -> nome exibido na planilha + nº de atributos
# (preencher o nº de atributos conforme sua base)
info_bases = {
    'HOG_128_16x16.csv':        ('HOG_128_16x16',        None),
    'HOG_128_32x32.csv':        ('HOG_128_32x32',        None),
    'HOG_256_32x32.csv':        ('HOG_256_32x32',        None),
    'LBP_256_3R.csv':           ('LBP_256_3R',           None),
    'LBP_256_6R.csv':           ('LBP_256_6R',           None),
    'LBP_256_12R.csv':          ('LBP_256_12R',          None),
    'PCA_075_HOG_128_16x16.csv':('PCA_075_HOG_128_16x16', None),
    'PCA_075_HOG_256_8x8.csv':  ('PCA_075_HOG_256_8x8',  None),
    'PCA_075_HOG_256_16x16.csv':('PCA_075_HOG_256_16x16', None),
    'PCA_075_HOG_256_32x32.csv':('PCA_075_HOG_256_32x32', None),
    'PCA_090_HOG_128_16x16.csv':('PCA_090_HOG_128_16x16', None),
    'PCA_090_HOG_256_32x32.csv':('PCA_090_HOG_256_32x32', None),
}

# Detectar nº de atributos automaticamente dos CSVs
for nome_base in bases:
    try:
        df_tmp = pd.read_csv(f'{BASE_PATH}{nome_base}', nrows=1)
        n_atr = df_tmp.shape[1] - 1  # -1 para a coluna classe
        nome_exib, _ = info_bases[nome_base]
        info_bases[nome_base] = (nome_exib, n_atr)
    except:
        pass

print('Informações das bases:')
for k, v in info_bases.items():
    print(f'  {v[0]:30s}  →  {v[1]} atributos')

Informações das bases:
  HOG_128_16x16                   →  1764 atributos
  HOG_128_32x32                   →  324 atributos
  HOG_256_32x32                   →  1764 atributos
  LBP_256_3R                      →  26 atributos
  LBP_256_6R                      →  50 atributos
  LBP_256_12R                     →  98 atributos
  PCA_075_HOG_128_16x16           →  105 atributos
  PCA_075_HOG_256_8x8             →  320 atributos
  PCA_075_HOG_256_16x16           →  199 atributos
  PCA_075_HOG_256_32x32           →  86 atributos
  PCA_090_HOG_128_16x16           →  210 atributos
  PCA_090_HOG_256_32x32           →  175 atributos


In [6]:
# Construir tabela resumida
linhas = []
for idx, nome_base in enumerate(bases, start=1):
    if nome_base not in resultados_globais:
        continue
    r = resultados_globais[nome_base]
    cv_scores = r['cv_scores']
    top10 = r['top10']
    nome_exib, n_atr = info_bases[nome_base]

    # Holdout scores para as top 10
    hold_scores = top10['f1_holdout'].tolist()

    # Linha 10-fold CV
    linha_cv = {'#': idx, 'Base': f'{nome_exib} ({n_atr} atributos)', 'Treinamento/Teste': '10-fold CV'}
    for i, s in enumerate(cv_scores, start=1):
        linha_cv[f'Conf. {i:02d}'] = round(s, 4)
    linhas.append(linha_cv)

    # Linha 70/30
    linha_ho = {'#': '', 'Base': '', 'Treinamento/Teste': '70/30'}
    for i, s in enumerate(hold_scores, start=1):
        linha_ho[f'Conf. {i:02d}'] = round(s, 4)
    linhas.append(linha_ho)

df_resultado = pd.DataFrame(linhas)

# Médias e desvios
conf_cols = [f'Conf. {i:02d}' for i in range(1, N_TOP+1)]
cv_rows  = df_resultado[df_resultado['Treinamento/Teste'] == '10-fold CV'][conf_cols]
ho_rows  = df_resultado[df_resultado['Treinamento/Teste'] == '70/30'][conf_cols]

medias_cv  = cv_rows.mean().round(4)
desvios_cv = cv_rows.std().round(4)
medias_ho  = ho_rows.mean().round(4)
desvios_ho = ho_rows.std().round(4)

# Adicionar linhas de rodapé
df_resultado = pd.concat([
    df_resultado,
    pd.DataFrame([{'#': '#', 'Base': 'Média (10-fold CV) =>', 'Treinamento/Teste': '', **medias_cv.to_dict()}]),
    pd.DataFrame([{'#': '#', 'Base': 'Desv. Pad. (10-fold CV) =>', 'Treinamento/Teste': '', **desvios_cv.to_dict()}]),
    pd.DataFrame([{'#': '#', 'Base': 'Média (70/30) =>', 'Treinamento/Teste': '', **medias_ho.to_dict()}]),
    pd.DataFrame([{'#': '#', 'Base': 'Desv. Pad. (70/30) =>', 'Treinamento/Teste': '', **desvios_ho.to_dict()}]),
], ignore_index=True)

print('\n=== TABELA MLP — F1 Score ===')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
print(df_resultado.to_string(index=False))


=== TABELA MLP — F1 Score ===
 #                                  Base Treinamento/Teste  Conf. 01  Conf. 02  Conf. 03  Conf. 04  Conf. 05  Conf. 06  Conf. 07  Conf. 08  Conf. 09  Conf. 10
 1        HOG_128_16x16 (1764 atributos)        10-fold CV    0.6719    0.6719    0.6719    0.6719    0.6719    0.6832    0.6862    0.6862    0.6862    0.6983
                                                     70/30    0.7346    0.7346    0.7346    0.7346    0.7346    0.7319    0.7319    0.7319    0.7319    0.7313
 2         HOG_128_32x32 (324 atributos)        10-fold CV    0.6129    0.6129    0.6129    0.6129    0.6129    0.7103    0.6996    0.6996    0.7096    0.7011
                                                     70/30    0.7296    0.7296    0.7296    0.7296    0.7296    0.7243    0.7243    0.7243    0.7238    0.7227
 3        HOG_256_32x32 (1764 atributos)        10-fold CV    0.6971    0.6971    0.6971    0.6971    0.6971    0.6936    0.6936    0.6936    0.6936    0.6936
               

## 4. Imprimir as 10 melhores configurações por base (Figura 3)

In [7]:
for nome_base in bases:
    if nome_base not in resultados_globais:
        continue
    nome_exib, n_atr = info_bases[nome_base]
    top10 = resultados_globais[nome_base]['top10']
    print(f'\n--- {nome_exib} ({n_atr} atributos) ---')
    for i, row in top10.iterrows():
        print(
            f"{i+1:2d} | F1_score: {row['f1_holdout']:.3f}; "
            f"hidden: {row['hls']}; "
            f"activation = {row['activation']}; "
            f"solver = {row['solver']}; "
            f"learning = {row['learning_rate']}; "
            f"max_iter = {int(row['max_iter'])}"
        )


--- HOG_128_16x16 (1764 atributos) ---
 1 | F1_score: 0.735; hidden: (70, 80); activation = relu; solver = sgd; learning = 0.1; max_iter = 1000
 2 | F1_score: 0.735; hidden: (70, 80); activation = relu; solver = sgd; learning = 0.1; max_iter = 1500
 3 | F1_score: 0.735; hidden: (70, 80); activation = relu; solver = sgd; learning = 0.1; max_iter = 2000
 4 | F1_score: 0.735; hidden: (70, 80); activation = relu; solver = sgd; learning = 0.1; max_iter = 2500
 5 | F1_score: 0.735; hidden: (70, 80); activation = relu; solver = sgd; learning = 0.1; max_iter = 3000
 6 | F1_score: 0.732; hidden: (70, 80); activation = relu; solver = sgd; learning = 0.001; max_iter = 1500
 7 | F1_score: 0.732; hidden: (70, 80); activation = relu; solver = sgd; learning = 0.001; max_iter = 2000
 8 | F1_score: 0.732; hidden: (70, 80); activation = relu; solver = sgd; learning = 0.001; max_iter = 2500
 9 | F1_score: 0.732; hidden: (70, 80); activation = relu; solver = sgd; learning = 0.001; max_iter = 3000
10 | F1

## 5. Exportar resultados para CSV (opcional)

In [8]:
df_resultado.to_csv('resultados_MLP.csv', index=False, encoding='utf-8-sig')
print('Arquivo resultados_MLP.csv salvo.')

Arquivo resultados_MLP.csv salvo.
